In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smis

/bin/bash: line 1: nvidia-smis: command not found


In [3]:
%%capture
# For Qwen3-VL
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

# Configuration

In [6]:
SEED = 42
USERNAME = 'alxxtexxr'

# Model configuration
SFT_LORA_ID = 'alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v260426081652'
RESUME_SFT_LORA_ID = None
RESUME_CKPT_STEP = None
# RESUME_SFT_LORA_ID = 'alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-GRPO-QLoRA-v260427113111'
# RESUME_CKPT_STEP = 25

# Data configuration
RL_DATA_ID = 'alxxtexxr/ViRL1.25K'

# Training configuration
MINI_BATCH_SIZE = 2
GRAD_ACC_STEPS = 4
NUM_EPOCHS = 1
WARMUP_STEPS = 5
LR = 5e-6

In [7]:
from datetime import datetime

# Resume training configuration
resume_from_checkpoint = bool(RESUME_SFT_LORA_ID)
if resume_from_checkpoint:
    model_name = RESUME_SFT_LORA_ID
    project_name = model_name.split('/')[-1]
    hub_model_id = RESUME_SFT_LORA_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_CKPT_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist!"
else:
    model_name = SFT_LORA_ID
    project_name = model_name.split('/')[-1]
    project_name = (
        f'{project_name.split("-SFT-QLoRA-v")[0] if "-SFT-QLoRA-v" in project_name else project_name}'
        f'-GRPO-QLoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    )
    hub_model_id = f'{USERNAME}/{project_name}'
    
print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", model_name)
print("Project name:", project_name)
print("Hugging Face model ID:", hub_model_id)

Resume from checkpoint: False
Model name: alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v260426081652
Project name: Qwen3-VL-8B-Thinking-VCB8K-GRPO-QLoRA-v260428214149
Hugging Face model ID: alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-GRPO-QLoRA-v260428214149


In [8]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print("Set random seed:", seed)

set_seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Set random seed: 42


In [9]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [10]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = resume_from_checkpoint if isinstance(resume_from_checkpoint, str) else model_name,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # dtype = torch.float16, # Force FP16
)

==((====))==  Unsloth 2026.4.8: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.92G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/5.99G [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/175M [00:00<?, ?B/s]

# Data

In [11]:
from datasets import load_dataset

dataset = load_dataset(RL_DATA_ID)
train_set = dataset['train']
val_set = dataset['val']

print("Train set:")
print(train_set)
print("\nValidation set:")
print(val_set)

README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/37.0M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/6.96M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/5.09M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/125 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/125 [00:00<?, ? examples/s]

Train set:
Dataset({
    features: ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase'],
    num_rows: 125
})


In [12]:
# Resize the images to (512, 512)
def resize_images(example):
    image = example["image"]
    image = image.resize((512, 512))
    example["decoded_image"] = image
    return example
train_set = train_set.map(resize_images)
val_set = val_set.map(resize_images)

# Convert the images to RGB
def convert_to_rgb(example):
    image = example["decoded_image"]
    if image.mode != "RGB":
        image = image.convert("RGB")
    example["decoded_image"] = image
    return example
train_set = train_set.map(convert_to_rgb)
val_set = val_set.map(convert_to_rgb)

# Clean the questions
def clean_question(example):
    question = example["question"]
    question = question.replace("<image>", "")    # Remove "<image>" from the question text
    question = " ".join(question.strip().split()) # Remove extra whitespace
    example["question"] = question
    return example

train_set = train_set.map(clean_question)
val_set = val_set.map(clean_question)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

In [13]:
THINK_START = "<think>"
THINK_END = "</think>"
VTHINK_START = "<vthink>"
VTHINK_END = "</vthink>"
ANSWER_START = "<answer>"
ANSWER_END = "</answer>"

In [14]:
def make_conversation(example):
    # Define placeholder constants if they are not defined globally
    # The user's text prompt
    text_content = (
        f"Question: {example['question']}\n\n"
        # f"Please first reason step by step using visual representations between {VTHINK_START} and {VTHINK_END}, "
        # f"then reason step by step in natural language between {THINK_START} and {THINK_END}, "
        # "and finally provide your final answer in LaTeX using \\boxed{} "
        # f"between {ANSWER_START} and {ANSWER_END}"
        f"Please reason through the problem step by step, using visual representations between {VTHINK_START} and {VTHINK_END} "
        f"and natural language between {THINK_START} and {THINK_END}. "
        "Provide your final answer in LaTeX using \\boxed{} "
        f"between {ANSWER_START} and {ANSWER_END}."
    )

    # Construct the prompt in the desired multi-modal format
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # Placeholder for the image
                {"type": "text", "text": text_content},  # The text part of the prompt
            ],
        },
    ]
    # The actual image data is kept separate for the processor
    return {"prompt": prompt, "image": example["decoded_image"], "answer": example["answer"]}

train_dataset = train_set.map(make_conversation)
val_dataset = val_set.map(make_conversation)

# We're reformatting dataset like this because decoded_images are the actual images
# The "image": example["decoded_image"] does not properly format the dataset correctly

# 1. Remove the original 'image' column
train_dataset = train_dataset.remove_columns("image")
val_dataset = val_dataset.remove_columns("image")

# 2. Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column("decoded_image", "image")
val_dataset = val_dataset.rename_column("decoded_image", "image")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

In [15]:
image = train_dataset[123]["image"]
prompt = train_dataset[123]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

<|im_start|>user
<|vision_start|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|ima

# Reward Functions

In [16]:
import re

THINK_PATTERN = re.compile(rf"{THINK_START}(.*?){THINK_END}", re.DOTALL)
VTHINK_PATTERN = re.compile(rf"{VTHINK_START}(.*?){VTHINK_END}", re.DOTALL)
ANSWER_PATTERN = re.compile(rf"{ANSWER_START}(.*?){ANSWER_END}", re.DOTALL)
VTOK_ID_PATTERN = re.compile(r'<\|vtok_(\d+)\|>')

def formatting_reward_func(completions, **kwargs) -> list[float]:
    LVL1_REWARD = 1.0
    LVL2_REWARD = 1.0
    LVL2_PENALTY = -0.3 #-0.9
    LVL1_PENALTY = -0.1575 #-0.162
    TOTAL_REWARD = 4.0
    
    scores = []
    
    for i, completion in enumerate(completions):
        # print("[TEST] Completion #"+str(i))
        score = 0
        
        if isinstance(completion, list):
            completion = completion[0]['content'] if completion else ""
        
        think_matches = THINK_PATTERN.findall(completion)
        vthink_matches = VTHINK_PATTERN.findall(completion)
        answer_matches = ANSWER_PATTERN.findall(completion)
        
        if len(answer_matches) == 1:
            # print("[TEST] Reward 1:", LVL1_REWARD)
            score += LVL1_REWARD
            # if answer_matches[0].strip().startswith('\\boxed{'):
            #    print("[TEST] Reward 1.5:", LVL2_REWARD)
            #     score += LVL2_REWARD
            # else:
            #    print("[TEST] Reward 1.5:", LVL2_PENALTY)
            #     score += LVL2_PENALTY
        else:
            # print("[TEST] Reward 1:", LVL1_PENALTY)
            score += LVL1_PENALTY
        
        if len(think_matches) > 0:
            # print("[TEST] Reward 2:", LVL1_REWARD)
            score += LVL1_REWARD
        else:
            # print("[TEST] Reward 2:", LVL1_PENALTY)
            score += LVL1_PENALTY
        
        vthink_count = len(vthink_matches)
        if vthink_count > 0:
            # print("[TEST] Reward 3:", LVL1_REWARD)
            score += LVL1_REWARD
            
            # Check if vthinking contains only valid visual tokens
            for vthink_match in vthink_matches:
                vthink_text = vthink_match.strip()
                
                vtok_ids = VTOK_ID_PATTERN.findall(vthink_text)        # Extract all token IDs
                cleaned = VTOK_ID_PATTERN.sub("", vthink_text).strip() # Remove them, check remainder
                valid = (
                    bool(vtok_ids) and
                    all(0 <= int(i) <= 8191 for i in vtok_ids) and
                    not cleaned
                )
                
                if valid:
                    # print("[TEST] Reward 4:", LVL2_REWARD / vthink_count)
                    score += LVL2_REWARD / vthink_count
                else:
                    # print("[TEST] Reward 4:", LVL2_PENALTY / vthink_count)
                    score += LVL2_PENALTY / vthink_count
        else:
            # print("[TEST] Reward 3:", LVL1_PENALTY)
            score += LVL1_PENALTY
        
        # Check for visual tokens outside of vthinking
        outside_vthink_text = re.sub(f"{VTHINK_START}.*?{VTHINK_END}", "", completion, flags=re.DOTALL)
        # if '<|vtok_' not in outside_vthink_text:
        #    print("[TEST] Reward 5:", LVL1_REWARD)
        #     score += LVL1_REWARD
        # else:
        #     # outside_vtok_count = len(VTOK_ID_PATTERN.findall(outside_vthink_text))
        #     # penalty_5 = max(LVL1_PENALTY * 0.1 * outside_vtok_count, LVL1_PENALTY) # Cap the penalty to LVL1_PENALTY
        #     penalty_5 = LVL1_PENALTY
            
        #    print("[TEST] Reward 5:", penalty_5)
        #     score += penalty_5
        
        if '<|vtok_' in outside_vthink_text:
            # outside_vtok_count = len(VTOK_ID_PATTERN.findall(outside_vthink_text))
            # penalty_5 = max(LVL1_PENALTY * 0.1 * outside_vtok_count, LVL1_PENALTY) # Cap the penalty to LVL1_PENALTY
            penalty_5 = LVL1_PENALTY
            
            # print("[TEST] Reward 5:", penalty_5)
            score += penalty_5

        # Fix up addCriterion issues
        # See https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks
        # Penalize on excessive addCriterion and newlines
        if len(completion) != 0:
            removal = completion.replace("addCriterion", "").replace("\n", "")
            removal_ratio = (len(completion) - len(removal)) / len(completion)
            # if removal_ratio < 0.5:
            #    print("[TEST] Reward 6:", LVL1_REWARD)
            #     score += LVL1_REWARD
            # else:
            #    print("[TEST] Reward 6:", LVL1_PENALTY * removal_ratio)
            #     score += LVL1_PENALTY * removal_ratio # Penalize proportionally to the excessiveness of addCriterion/newlines
            
            if removal_ratio > 0.5:
                # print("[TEST] Reward 6:", LVL1_PENALTY * removal_ratio)
                score += LVL1_PENALTY * removal_ratio # Penalize proportionally to the excessiveness of addCriterion/newlines
            
        # Normalize score; penalties are intentionally asymmetric —
        # individual penalties are kept mild to avoid canceling out earned rewards.
        # print("[TEST] Total score:", score)
        score /= TOTAL_REWARD
        # print("[TEST] Normalized score:", score)

        scores.append(score)
        
        # print("[TEST] ================================\n")
        
    return scores

formatting_reward_func([
# 0. Good example
"""<vthink><|vtok_0|><|vtok_8191|></vthink>

<think>Hmm...</think>

<answer>\\boxed{Yes}</answer>""",

# 1. Bad example: non-visual tokens in vthink
"""<vthink><|vtok_0|>Hello, world!<|vtok_8191|></vthink>

<think>
Some reasoning.
</think>

<answer>\\boxed{Yes}</answer>""",

# 2. Bad example: visual tokens exceeding the valid range
"""<vthink><|vtok_0|><|vtok_8192|></vthink>

<think>Hmm...</think>

<answer>\\boxed{Yes}</answer>""",

# 3. Bad example: visual tokens outside of vthink
"""<vthink><|vtok_0|><|vtok_8191|></vthink>

<think>Hmm...</think>

<|vtok_1|><|vtok_2|>

<answer>\\boxed{Yes}</answer>""",

# 4. Bad example: doesn't contain vthink
"""<think>Hmm...</think>

<answer>\\boxed{Yes}</answer>""",

# 5. Bad example: doesn't contain think
"""<vthink><|vtok_0|><|vtok_8191|></vthink>

<answer>\\boxed{Yes}</answer>""",

# 6. Bad example: doesn't use answer tag
"""<vthink><|vtok_0|><|vtok_8191|></vthink>

<think>Hmm...</think>

\\boxed{Yes}""",

# 6. Bad example: doesn't use all necessary tags
"""\\boxed{Yes}""",
])

[1.0,
 0.675,
 0.675,
 0.960625,
 0.460625,
 0.7106250000000001,
 0.7106250000000001,
 -0.11812500000000001]

In [17]:
import re
from collections import Counter

BOXED_PATTERN = re.compile(r"\\boxed{(.*?)}", re.DOTALL)

def normalize_whitespace(s):
    return " ".join(s.split())

def remove_boxeds(s):
    return normalize_whitespace(BOXED_PATTERN.sub(r"\1 ", s))

def remove_nums(s):
    return normalize_whitespace(re.sub(r"\d+", "", s))

def match_nums(s, t):
    nums_s = re.findall(r"\d+", s)
    nums_t = re.findall(r"\d+", t)
    return nums_s == nums_t or Counter(nums_s) == Counter(nums_t)

def cheap_tokenizer(t):
    return re.findall(r"\d+|[a-z]+", t)

def compute_token_reward(s, t, reward, penalty):    
    assert reward > 0 and penalty < 0, "Reward and penalty must be positive and negative, respectively!"
    
    s, t = cheap_tokenizer(s), cheap_tokenizer(t)

    if not len(s) and not len(t):
        return reward
    
    # matched = tokens in both a and b (intersection)
    # extra   = tokens in a not in b (penalized)
    matched = list((Counter(s) & Counter(t)).elements())
    extra   = list((Counter(s) - Counter(t)).elements())
    
    total_reward = len(matched) * (reward / len(t)) if len(t) else 0
    total_penalty = len(extra) * (penalty / len(s)) if len(s) else 0
    
    return total_reward + total_penalty

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    TOTAL_REWARD = 3.0
    LVL1_PENALTY = -0.15
    
    completions = [(c[0]['content'] if c else "") if isinstance(c, list) else c for c in completions]
    responses = [ANSWER_PATTERN.findall(completion) for completion in completions]

    scores = []
    for response, answer_ in zip(responses, answer):
        if len(response) != 1:
            # print("[TEST] Reward 0:", 0.0)
            scores.append(0.0)
            continue
        
        if response[0] == answer_:
            # print("[TEST] Reward 0:", 1.0)
            scores.append(1.0)
            continue

        response = response[0]
        score = 0.0
        
        boxed_matches = BOXED_PATTERN.findall(response)
        if len(boxed_matches):
            boxed_reward = 0.5 / len(boxed_matches)
            # print("[TEST] Reward 1:", boxed_reward)
            score += boxed_reward
        else:
            # print("[TEST] Reward 1:", LVL1_PENALTY)
            score += LVL1_PENALTY
            
        response = remove_boxeds(response)
        answer_ = remove_boxeds(answer_)
        
        if match_nums(response, answer_):
            # print("[TEST] Reward 2:", 2.0)
            score += 2.0
        else:
            # print("[TEST] Reward 2:", LVL1_PENALTY)
            score += LVL1_PENALTY
            
        response = remove_nums(response)
        answer_ = remove_nums(answer_)
        
        token_reward = compute_token_reward(response, answer_, reward=0.5, penalty=LVL1_PENALTY)
        # print("[TEST] Reward 3:", token_reward)
        score += token_reward
        
        score /= TOTAL_REWARD

        scores.append(score)
    
    predicted = responses[0][0] if responses[0] else "N/A"
    print("###### Rewards:")
    print(scores)
    print("\n###### Prompt #1:")
    print(prompts[0][0]["content"][1]["text"])
    print("\n###### True Answer #1:")
    print(answer[0])
    print("\n###### Predicted Answer #1:")
    print(predicted)
    print("\n###### Full Response #1:")
    print(completions[0])
    print("\n================================================================================================================================\n")

    return scores

# # Good example: the predicted answer matches the true answer exactly
# correctness_reward_func(
#     [[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': "Question: In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:\n\nPlease first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>"}]}]],
#     ["<answer>\\boxed{180}</answer>"],
#     ["\\boxed{180}"],
# )

# # Good example: the predicted answer has spaces but matches the true answer
# correctness_reward_func(
#     [[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': "Question: In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:\n\nPlease first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>"}]}]],
#     ["<answer>\\boxed{ 180 }</answer>"],
#     ["\\boxed{180}"],
# )

# # Bad example: the predicted answer has matched numbers but doesn't use \boxed{}
# correctness_reward_func(
#     [[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': "Question: In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:\n\nPlease first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>"}]}]],
#     ["<answer>180</answer>"],
#     ["\\boxed{180}"],
# )

# Bad example: the predicted answer contains the true answer but use extra unnecessary tokens
correctness_reward_func(
    [[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': "Question: In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:\n\nPlease first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>"}]}]],
    ["<answer>The answer is \\boxed{180}</answer>"],
    ["\\boxed{180}"],
)

# # Bad example: the predicted answer contains the true answer but does not contain extra tokens from the true answer
# correctness_reward_func(
#     [[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': "Question: In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:\n\nPlease first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>"}]}]],
#     ["<answer>\\boxed{180}</answer>"],
#     ["The answer is \\boxed{180}"],
# )

# # Bad example: the predicted answer numbers don't match those of the true answer exactly
# correctness_reward_func(
#     [[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': "Question: In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:\n\nPlease first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>"}]}]],
#     ["<answer>\\boxed{3 - 2 = b}</answer>"],
#     ["\\boxed{3 - 1 = b}"],
# )

###### Rewards:
[0.7833333333333333]

###### Prompt #1:
Question: In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:

Please first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \boxed{} between <answer> and </answer>

###### True Answer #1:
\boxed{180}

###### Predicted Answer #1:
The answer is \boxed{180}

###### Full Response #1:
<answer>The answer is \boxed{180}</answer>




[0.7833333333333333]

# Training

In [18]:
import math
from trl import GRPOConfig, GRPOTrainer

max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACC_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

training_args = GRPOConfig(
    # Training arguments
    seed = SEED,
    
    per_device_train_batch_size = MINI_BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACC_STEPS,
    num_generations = 4, # Decrease if out of memory
    
    max_prompt_length = 1024,
    max_completion_length = 2048,
    
    # max_steps = 50, # For testing
    max_steps = max_steps,
    # num_train_epochs = NUM_EPOCHS,
    # warmup_ratio=0.05,
    warmup_steps = WARMUP_STEPS,
    learning_rate = LR,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    max_grad_norm = 0.1,
    weight_decay = 0.01,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps = 1,
    log_completions = False,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    # save_steps=1, # For testing
    save_steps=5,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. Warning: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    # load_best_model_at_end=True, 

    output_dir=project_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,

    hub_strategy='all_checkpoints',
    hub_always_push=True,

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)

Calculated max steps: 125


In [19]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
        formatting_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
)

trainer.train(resume_from_checkpoint=resume_from_checkpoint)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,875,692,272 (0.49% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`generation_config` default values have been modified to match model-specific defaults: {'top_p': 0.95}. If this is not desired, please set these values explicitly.


###### Rewards:
[0.06666666666666667, 0.7833333333333333, 0.2833333333333333, 0.0, 0.2833333333333333, 1.0, 1.0, 1.0]

###### Prompt #1:
Question: How many students lifted more than 60 pounds?

Please reason through the problem step by step, using visual representations between <vthink> and </vthink> and natural language between <think> and </think>. Provide your final answer in LaTeX using \boxed{} between <answer> and </answer>.

###### True Answer #1:
\boxed{8}

###### Predicted Answer #1:
6

###### Full Response #1:
The image shows a stem-and-leaf plot with stems ranging from 3 to 9 and corresponding leaves. To determine how many students lifted more than 60 pounds, we focus on the stems above 6.

For the stem 7, the image shows leaves 2 and 5, representing 72 and 75 pounds.
For the stem 8, the image shows leaves 2, 6, and 6, representing 82, 86, and 86 pounds.
For the stem 9, the image shows leaf 0, representing 90 pounds.

Adding these up, we have 2 (from 7) + 3 (from 8) + 1 (fro

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / formatting_reward_func / mean,rewards / formatting_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,-0.043100,0.847943,0.410615,585.625000,358.000000,928.000000,0.000000,585.625000,358.000000,928.000000,0.913423,0.295859,0.126032,0.552083,0.437565


###### Rewards:
[1.0, 0.06666666666666667, 0.7833333333333333, 0.0, 0.0, 1.0, 0.0, 0.0]

###### Prompt #1:
Question: If you subtract 1 brown ball, how many objects remain?

Please reason through the problem step by step, using visual representations between <vthink> and </vthink> and natural language between <think> and </think>. Provide your final answer in LaTeX using \boxed{} between <answer> and </answer>.

###### True Answer #1:
\boxed{9}

###### Predicted Answer #1:
\boxed{9}

###### Full Response #1:
Okay, so I'm trying to figure out how many objects remain after subtracting one brown ball from the scene. Let me start by looking at the image directly.

The image shows a 3D scene with various objects. As seen in the image:

- There is a large silver metallic cylinder in the center.
- Around it, there are several other objects:
  - A small silver metallic cylinder to the left.
  - A small red sphere to the left of the large cylinder.
  - A small purple sphere to the right of the r

: 

# Inference

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

image = None
instruction = "Several slices of bread with side salads on a table."

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

In [ ]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)